Thoroughly document data preprocessing pipelines and transformation steps in a reproducible and automated manner, using tools like Jupyter Notebooks and data lineage tracking for governance compliance.

In [1]:
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime

In [2]:
np.random.seed(42)
n_samples = 1500

In [3]:
raw_df = pd.DataFrame({
    'patient_id': [f"PT-{10000 + i}" for i in range(n_samples)],
    'age': np.random.normal(62, 14, n_samples).round(1),
    'troponin': np.random.lognormal(-1.5, 0.9, n_samples).round(4),
    'creatinine': np.random.normal(1.1, 0.4, n_samples).round(2),
    'systolic_bp': np.random.normal(134, 18, n_samples).round(0),
    'admission_type': np.random.choice(['Emergency', 'Elective', 'Urgent'], n_samples, p=[0.5, 0.3, 0.2]),
    'target': np.random.binomial(1, 0.35, n_samples)
})

In [4]:
raw_df.loc[raw_df.sample(frac=0.05, random_state=42).index, 'creatinine'] = np.nan
raw_df.loc[raw_df.sample(frac=0.03, random_state=7).index, 'troponin'] = np.nan
raw_df.loc[0, 'systolic_bp'] = 410.0  # Sensor artifact outlier

In [5]:
def compute_dataframe_hash(df: pd.DataFrame) -> str:
    row_hashes = pd.util.hash_pandas_object(df, index=True).values
    return hashlib.sha256(row_hashes.tobytes()).hexdigest()

In [6]:
raw_data_hash = compute_dataframe_hash(raw_df)
print(f"Raw Dataset Shape : {raw_df.shape}")
print(f"Raw Lineage SHA-256: {raw_data_hash[:16]}...")

Raw Dataset Shape : (1500, 7)
Raw Lineage SHA-256: 79f81d6672af0b9e...


In [7]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [8]:
class PhysiologicalClipper(BaseEstimator, TransformerMixin):
    """
    Domain-specific outlier clipper that enforces physiological bounds
    and logs exact clip counts for regulatory auditability.
    """
    def __init__(self, bounds: dict):
        self.bounds = bounds
        self.clip_audit_log_ = {}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        self.clip_audit_log_ = {}
        for col, (low, high) in self.bounds.items():
            if col in X_out.columns:
                n_low = int((X_out[col] < low).sum())
                n_high = int((X_out[col] > high).sum())
                X_out[col] = X_out[col].clip(lower=low, upper=high)
                self.clip_audit_log_[col] = {
                    "lower_bound": low,
                    "upper_bound": high,
                    "values_clipped_low": n_low,
                    "values_clipped_high": n_high
                }
        return X_out

In [9]:
id_col = 'patient_id'
target_col = 'target'
num_features = ['age', 'troponin', 'creatinine', 'systolic_bp']
cat_features = ['admission_type']

In [10]:
clinical_bounds = {
    'age': (18.0, 110.0),
    'troponin': (0.0, 50.0),
    'creatinine': (0.2, 15.0),
    'systolic_bp': (60.0, 250.0)
}

In [11]:
numeric_pipeline = Pipeline([
    ('median_imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('robust_scaler', RobustScaler())
])

In [12]:
categorical_pipeline = Pipeline([
    ('mode_imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [13]:
column_transformer = ColumnTransformer([
    ('num_block', numeric_pipeline, num_features),
    ('cat_block', categorical_pipeline, cat_features)
], remainder='drop')

In [14]:
preprocessing_pipeline = Pipeline([
    ('physiological_clipper', PhysiologicalClipper(bounds=clinical_bounds)),
    ('feature_transformer', column_transformer)
])

In [15]:
X_raw = raw_df[num_features + cat_features]
X_transformed = preprocessing_pipeline.fit_transform(X_raw)
output_feature_names = preprocessing_pipeline.named_steps['feature_transformer'].get_feature_names_out().tolist()

In [16]:
df_processed = pd.DataFrame(X_transformed, columns=output_feature_names)
processed_data_hash = compute_dataframe_hash(df_processed)

In [17]:
print(f"Transformed Matrix Shape : {df_processed.shape}")
print(f"Output Lineage SHA-256   : {processed_data_hash[:16]}...")

Transformed Matrix Shape : (1500, 9)
Output Lineage SHA-256   : ccc22792a4e88e35...


In [18]:
import json
import sklearn

In [19]:
clipper_step = preprocessing_pipeline.named_steps['physiological_clipper']
ct_step = preprocessing_pipeline.named_steps['feature_transformer']

In [20]:
num_imputer = ct_step.named_transformers_['num_block'].named_steps['median_imputer']
num_scaler = ct_step.named_transformers_['num_block'].named_steps['robust_scaler']
cat_encoder = ct_step.named_transformers_['cat_block'].named_steps['onehot_encoder']

In [21]:
lineage_manifest = {
    "governance_metadata": {
        "pipeline_name": "clinical_diagnostic_preprocessing_v1",
        "execution_timestamp": datetime.now().isoformat(),
        "sklearn_version": sklearn.__version__,
        "random_seed": 42
    },
    "data_lineage": {
        "source_table": "raw_clinical_intake",
        "source_row_count": int(len(raw_df)),
        "source_sha256": raw_data_hash,
        "dropped_pii_columns": [id_col],
        "output_row_count": int(len(df_processed)),
        "output_column_count": int(df_processed.shape[1]),
        "output_sha256": processed_data_hash
    },
    "transformation_steps": [
        {
            "step_order": 1,
            "step_name": "Physiological Bound Clipping",
            "columns_affected": num_features,
            "audit_details": clipper_step.clip_audit_log_
        },
        {
            "step_order": 2,
            "step_name": "Numeric Median Imputation + Missingness Indicators",
            "columns_affected": num_features,
            "learned_medians": {col: round(float(val), 4) for col, val in zip(num_features, num_imputer.statistics_)}
        },
        {
            "step_order": 3,
            "step_name": "Numeric Robust Scaling (IQR)",
            "columns_affected": num_features,
            "learned_centers": {col: round(float(val), 4) for col, val in zip(num_features, num_scaler.center_)},
            "learned_scales": {col: round(float(val), 4) for col, val in zip(num_features, num_scaler.scale_)}
        },
        {
            "step_order": 4,
            "step_name": "Categorical One-Hot Encoding",
            "columns_affected": cat_features,
            "learned_categories": {col: cats.tolist() for col, cats in zip(cat_features, cat_encoder.categories_)}
        }
    ],
    "output_schema": output_feature_names
}

In [22]:
lineage_file = "data_lineage_manifest.json"
with open(lineage_file, "w", encoding="utf-8") as f:
    json.dump(lineage_manifest, f, indent=4)

In [23]:
print(f"Governance lineage manifest exported to '{lineage_file}'.")

Governance lineage manifest exported to 'data_lineage_manifest.json'.


In [24]:
from IPython.display import display, Markdown

In [29]:
stats_lines = []
medians = lineage_manifest["transformation_steps"][1]["learned_medians"]
scales = lineage_manifest["transformation_steps"][2]["learned_scales"]
clips = lineage_manifest["transformation_steps"][0]["audit_details"]

In [30]:
for col in num_features:
    total_clips = clips[col]["values_clipped_low"] + clips[col]["values_clipped_high"]
    stats_lines.append(
        f"{col}: bounds [{clips[col]['lower_bound']}, {clips[col]['upper_bound']}], "
        f"clipped {total_clips}, median {medians[col]}, IQR scale {scales[col]}"
    )

In [33]:
stats_summary = "\n".join(stats_lines)

In [34]:
compliance_doc = f"""Data Preprocessing and Lineage Report
Pipeline: {lineage_manifest['governance_metadata']['pipeline_name']}
Run Time: {lineage_manifest['governance_metadata']['execution_timestamp']}
Environment: scikit-learn {lineage_manifest['governance_metadata']['sklearn_version']} (seed {lineage_manifest['governance_metadata']['random_seed']})

Data Lineage
Raw Source ({lineage_manifest['data_lineage']['source_table']}): {lineage_manifest['data_lineage']['source_row_count']} rows, {raw_df.shape[1]} columns, SHA-256 {raw_data_hash[:16]}...
Processed Output (df_processed): {lineage_manifest['data_lineage']['output_row_count']} rows, {lineage_manifest['data_lineage']['output_column_count']} columns, SHA-256 {processed_data_hash[:16]}...

Notes:
- Removed patient ID column ({id_col}) before processing.
- Added binary missingness indicator columns to track missing lab values.

Feature Transformations
{stats_summary}
admission_type: one-hot encoded categories {lineage_manifest['transformation_steps'][3]['learned_categories']['admission_type']}

Output Features ({len(output_feature_names)} total)
{', '.join(output_feature_names)}
"""

In [35]:
doc_file = "preprocessing_governance_report.txt"
with open(doc_file, "w", encoding="utf-8") as f:
    f.write(compliance_doc)

In [36]:
print(compliance_doc)

Data Preprocessing and Lineage Report
Pipeline: clinical_diagnostic_preprocessing_v1
Run Time: 2026-11-19T11:29:53.291828
Environment: scikit-learn 1.6.1 (seed 42)

Data Lineage
Raw Source (raw_clinical_intake): 1500 rows, 7 columns, SHA-256 79f81d6672af0b9e...
Processed Output (df_processed): 1500 rows, 9 columns, SHA-256 ccc22792a4e88e35...

Notes:
- Removed patient ID column (patient_id) before processing.
- Added binary missingness indicator columns to track missing lab values.

Feature Transformations
age: bounds [18.0, 110.0], clipped 2, median 62.7, IQR scale 18.2
troponin: bounds [0.0, 50.0], clipped 0, median 0.2238, IQR scale 0.2701
creatinine: bounds [0.2, 15.0], clipped 20, median 1.09, IQR scale 0.51
systolic_bp: bounds [60.0, 250.0], clipped 1, median 134.0, IQR scale 24.0
admission_type: one-hot encoded categories ['Elective', 'Emergency', 'Urgent']

Output Features (9 total)
num_block__age, num_block__troponin, num_block__creatinine, num_block__systolic_bp, num_block__m